# Record-level fixes — evidence-carrying patch ops

Every change is a patch op `{op, field, value, evidence span}` validated mechanically before it applies. Verified probe spans become `add` ops and whole-cell hits in non-text fields become `move` ops without an LLM; admitted records get one composed LLM call whose ops pass a validation gate (field whitelist, verbatim copy-check, whole-cell guard, tier-1 parser).

> **This notebook is the phase-1 probe, not the production run.** It runs a `SAMPLE_N` subset against a local `gpt-oss-20b` under task contract v1. The corpus-scale run is `mds_norm.pipeline.extraction` (`prepare` / `run` / `finalise`), which writes the same `record_patches.parquet`; the shared queues and gate are imported from there.

In [1]:
import json
import re
from pathlib import Path

import polars as pl
from codecarbon import EmissionsTracker
from mds_data_model.introspection import (
    free_text_fields,
    measurement_fields,
    monetary_fields,
)

from mds_norm.parsers.parse_dates import parse_date
from mds_norm.parsers.parse_dimensions import parse_dimensions
from mds_norm.parsers.parse_monetary import parse_monetary
from mds_norm.utils.inference import Inference

In [2]:
DATA_PATH = Path("../data/mds-flat-records.parquet")

INTERMEDIATE_PATH = Path(".").resolve() / "analysis_output"
FIELD_STATS = INTERMEDIATE_PATH / "field_stats.parquet"
PROBE_CANDIDATES = INTERMEDIATE_PATH / "probe_candidates.parquet"

OUT_DIR = INTERMEDIATE_PATH / "record_fixes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMISSIONS_LOG_PATH = INTERMEDIATE_PATH / "emissions_logs"
EMISSIONS_LOG_PATH.mkdir(parents=True, exist_ok=True)

COMPONENT = "record_fixes"
TIER = 5

LLM_BASE = "http://localhost:30000/v1"
LLM_MODEL = "gpt-oss-20b"
CONCURRENCY = 100
SAMPLE_N = 5000  # test subset; raise to run the full admission queue
RELOC_N = 1000   # test subset of the relocation class

TEXT_FIELDS = pl.col("field_type").is_in(free_text_fields())
MEASUREMENT_GROUPS = list(measurement_fields())

## 1. Patch contract

Destination field per probe group and the verifier each target must pass. Dates route by source-field context and default to production date.

In [3]:
from mds_norm.utils.patches import (DATE_DEST_BY_SOURCE, DEST, MATERIAL_MAX_WORDS, MIN_TEXT_CHARS, TARGET_PRESENCE,
                           MONETARY_FIELDS, PRICE_DEST, PROD_DATE, SYSTEM, LLM_CONFIDENCE, TASK_OF_FIELD, parse_ops, validate_op, build_request, strip_prefix, verify_value)


## 2. Mechanical ops from the probe queue

*refine*/*novel* spans in free-text fields become `add` ops; whole-cell hits in non-text fields become `move` ops; *additional* spans stay `flagged`. Partial-cell hits in non-text fields form the LLM relocation queue.

In [ ]:
from mds_norm.pipeline.extraction import probe_ops

with EmissionsTracker(project_name="record_fixes_mechanical", output_dir=str(EMISSIONS_LOG_PATH), log_level="error") as tracker:
    mechanical_patches, reloc_all = probe_ops()

mechanical_patches.group_by("op", "task", "status").len().sort("len", descending=True)

In [ ]:
reloc_q = (
    reloc_all
    .sort(pl.col("hits").list.len(), "record_id", descending=[True, False])
    .head(RELOC_N)
)
print(f"{len(reloc_q)} records admitted for relocation of {len(reloc_all)} (test cap {RELOC_N})")

## 3. LLM admission — extraction queue

Records with free-text mass whose target slots (production date, material, dimensions) are empty, ordered by missing targets then text mass. Any date field counts as date-populated.

In [ ]:
from mds_norm.pipeline.extraction import admission_queue

extract_q = (
    admission_queue()
    .sort("n_missing", "text_chars", descending=True)
    .head(SAMPLE_N)
)
print(f"{len(extract_q)} records admitted for extraction (test cap {SAMPLE_N})")
extract_q.head()

## 4. Record rendering and composed prompt

One call per record, rendered as nested JSON with prefixes stripped and repeats collapsed. Ops reference `source_field`; the gate maps it back to a node by containment. The system prompt requires `value` to be a verbatim contiguous substring.

In [ ]:
needs = {r["record_id"]: r for r in extract_q.iter_rows(named=True)}
reloc_hits = {r["record_id"]: r["hits"] for r in reloc_q.iter_rows(named=True)}
admitted = sorted(set(needs) | set(reloc_hits))

fs = pl.scan_parquet(FIELD_STATS)
flat = (
    pl.scan_parquet(DATA_PATH)
    .filter(
        pl.col("record_id").is_in(admitted)
        & pl.col("field_type").str.starts_with("spectrum/"))
    .select("record_id", "data_source", "node_id", "parent_id", "depth", "field_type")
    .join(fs.select("node_id", "value"), on="node_id", how="left")
    .collect(engine="streaming")
)
record_rows = {
    (key if isinstance(key, str) else key[0]): sub
    for key, sub in flat.partition_by("record_id", as_dict=True).items()
}


def request_for(record_id: str) -> dict | None:
    row = needs.get(record_id)
    missing = [k for k in ("date", "material", "dimension")
               if row is not None and not row[f"has_{k}"]]
    return build_request(record_id, record_rows[record_id].to_dicts(),
                         missing, reloc_hits.get(record_id, []))


requests = [req for rid in admitted if (req := request_for(rid)) is not None]
print(f"{len(requests)} requests")
print(requests[0]["prompt"][:700])


## 5. Async queries

`utils.inference.Inference` against the local server; greedy decoding, one retry, failures become deferrals. Raw responses and token usage are persisted.

In [8]:
inf = Inference(model=LLM_MODEL, base_url=LLM_BASE, concurrency=CONCURRENCY,
                timeout=180.0, retries=1)

with EmissionsTracker(project_name="record_fixes_llm", output_dir=str(EMISSIONS_LOG_PATH), log_level="error") as tracker:
    responses = await inf.generate(
        [r["prompt"] for r in requests],
        system=SYSTEM,
        usage=True,
        temperature=0.0,
        max_tokens=2048,
        reasoning_effort="low",
    )

raw = pl.DataFrame([{"record_id": r["record_id"], **resp}
                    for r, resp in zip(requests, responses)])
raw.write_parquet(OUT_DIR / "llm_responses.parquet")
print(f"{len(raw)} responses, {raw['error'].is_not_null().sum()} errors, "
      f"{raw['prompt_tokens'].sum():,} prompt / {raw['completion_tokens'].sum():,} completion tokens")

Output()

5996 responses, 0 errors, 5,706,389 prompt / 604,468 completion tokens


## 6. Validation gate → patch sidecar

Each op is checked for type, admitted field, source node by containment with span recovery, whole-cell guard and tier-1 verifier. Failures are kept as `rejected` rows; unparseable responses defer the whole record.

**Careful:** the next cell writes `record_patches.parquet`, the sidecar the compiler reads. Running it after a production pass replaces the full-queue sidecar with this subset's; re-run `mds_norm.pipeline.extraction finalise` to restore it.

In [14]:
llm_rows = []
for req, resp in zip(requests, responses):
    base = {"record_id": req["record_id"], "data_source": req["data_source"],
            "sub_component": "llm"}
    ops = parse_ops(resp["content"])
    if ops is None:
        reason = "llm_error" if resp["error"] else "unparseable_response"
        llm_rows.append(base | {"status": "deferred", "reason": reason})
        continue
    for op in ops:
        parsed, reason = validate_op(req, op)
        rationale = op.get("rationale") if isinstance(op, dict) else None
        if parsed is None:
            proposed = op if isinstance(op, dict) else {}
            llm_rows.append(base | {
                "op": proposed.get("op") if isinstance(proposed.get("op"), str) else None,
                "field": proposed.get("field") if isinstance(proposed.get("field"), str) else None,
                "value": proposed.get("value") if isinstance(proposed.get("value"), str) else None,
                "status": "rejected", "reason": reason, "rationale": rationale,
            })
        else:
            task = ("relocate" if parsed["op"] == "move"
                    else TASK_OF_FIELD.get(parsed["field"], "relocate"))
            llm_rows.append(base | parsed | {
                "task": task, "status": "resolved",
                "confidence": LLM_CONFIDENCE, "rationale": rationale,
            })

llm_patches = pl.from_dicts(llm_rows, schema_overrides={
    "node_id": pl.Binary, "span_start": pl.UInt32, "span_end": pl.UInt32,
    "confidence": pl.Float64})

# two composed tasks can propose one span; keep one
llm_patches = llm_patches.unique(
    subset=["record_id", "node_id", "op", "field", "value", "span_start"],
    keep="first", maintain_order=True)

patches = (
    pl.concat([mechanical_patches, llm_patches], how="diagonal_relaxed")
    .with_columns(component=pl.lit(COMPONENT), tier=pl.lit(TIER))
)
patches.write_parquet(OUT_DIR / "record_patches.parquet")
print(f"{len(patches):,} patch rows -> {OUT_DIR / 'record_patches.parquet'}")

659,171 patch rows -> /home/liam/Documents/university/mds-norm/notebooks/analysis_output/record_fixes/record_patches.parquet


## 7. Outcomes, cost, review sample

Ops per task and status, rejection reasons, tokens per admitted record and per accepted op. Review sample is head+tail over resolved LLM ops plus a mechanical sample.

In [10]:
display(patches.group_by("sub_component", "task", "op", "status").len()
      .sort("sub_component", "len", descending=[False, True]))

rejects = (
    llm_patches.filter(pl.col("status").is_in(["rejected", "deferred"]))
    .group_by("reason").len().sort("len", descending=True)
)
rejects

sub_component,task,op,status,len
str,str,str,str,u32
"""llm""","""extract_material""","""add""","""resolved""",3924
"""llm""","""relocate""","""move""","""resolved""",3104
"""llm""","""extract_date""","""add""","""resolved""",905
"""llm""",null,"""add""","""rejected""",675
"""llm""","""extract_dimension""","""add""","""resolved""",467
"""llm""",null,null,"""deferred""",1
"""mechanical""","""probe_refine""","""add""","""resolved""",279398
"""mechanical""","""probe_novel""","""add""","""resolved""",220552
"""mechanical""","""probe_additional""","""add""","""flagged""",149948


reason,len
str,u32
"""date_parse_failed""",318
"""dimension_parse_failed""",185
"""copy_check_failed""",124
"""material_too_long""",23
"""whole_cell_extraction""",13
"""material_has_digits""",10
"""empty_value""",2
"""unparseable_response""",1


In [11]:
accepted = llm_patches.filter(pl.col("status") == "resolved")
total_tokens = raw["prompt_tokens"].sum() + raw["completion_tokens"].sum()
print(f"accepted LLM ops: {len(accepted)} across {accepted['record_id'].n_unique()} records")
print(f"tokens/record: {total_tokens / len(requests):,.0f}   "
      f"tokens/accepted op: {total_tokens / max(len(accepted), 1):,.0f}")
print(f"records yielding no accepted op: "
      f"{len(requests) - accepted['record_id'].n_unique()} / {len(requests)}")

accepted LLM ops: 8400 across 2873 records
tokens/record: 1,053   tokens/accepted op: 751
records yielding no accepted op: 3123 / 5996


In [12]:
REVIEW_N = 20

review_cols = ["record_id", "sub_component", "task", "op", "source_field", "field",
               "value", "rationale", "confidence"]
review = pl.concat([
    accepted.head(REVIEW_N),
    accepted.tail(REVIEW_N),
    mechanical_patches.filter(pl.col("status") == "resolved")
    .sample(min(REVIEW_N, len(mechanical_patches)), seed=0),
], how="diagonal_relaxed").select(review_cols)

review.write_csv(OUT_DIR / "record_fixes_review_sample.csv")
review

record_id,sub_component,task,op,source_field,field,value,rationale,confidence
str,str,str,str,str,str,str,str,f64
"""00116cde-5269-3e40-8178-f33685…","""llm""","""extract_date""","""add""","""spectrum/brief_description""","""spectrum/object_production_dat…","""1901-1947""","""production period stated""",0.7
"""00116cde-5269-3e40-8178-f33685…","""llm""","""extract_material""","""add""","""spectrum/brief_description""","""spectrum/material""","""White metal""","""material mentioned""",0.7
"""00135101-b63f-3aab-8273-703907…","""llm""","""relocate""","""move""","""spectrum/technical_attribute""","""spectrum/dimension""","""63mm""","""misplaced dimension""",0.7
"""00135101-b63f-3aab-8273-703907…","""llm""","""relocate""","""move""","""spectrum/technical_attribute""","""spectrum/dimension""","""48mm""","""misplaced dimension""",0.7
"""00135101-b63f-3aab-8273-703907…","""llm""","""relocate""","""move""","""spectrum/technical_attribute""","""spectrum/dimension""","""25mm""","""misplaced dimension""",0.7
…,…,…,…,…,…,…,…,…
"""57e2a6df-f73f-3851-b87c-7c8d74…","""mechanical""","""probe_refine""","""add""","""spectrum/brief_description""","""spectrum/dimension""","""50mm""",null,0.9
"""224e6336-fcf5-3b21-9f2d-f8d654…","""mechanical""","""probe_novel""","""add""","""spectrum/brief_description""","""spectrum/technical_attribute_m…","""18 pages""",null,0.8
"""72eafd96-8c77-3739-9619-b9725f…","""mechanical""","""probe_novel""","""add""","""spectrum/brief_description""","""spectrum/object_production_dat…","""April 1894""",null,0.8
